# 🕳️ Pothole Detection — YOLOv8l Training (Colab ↔ Kaggle)

## 🚀 Single-Stage YOLOv8l Pipeline — Colab Edition

| Feature | Detail |
|----------|--------|
| **Environment** | Google Colab (12h session limit) |
| **Model** | YOLOv8l · 768px · 400 epochs |
| **Time Guard** | Auto-save before 11.5h limit |
| **Resume** | `last.pt` → `best.pt` → fresh (crash-safe) |
| **NaN Safety** | AMP disabled · lr0=3e-4 · gradient clip · loss guard |
| **Cross-platform** | Checkpoint folder compatible with Kaggle notebook |

> **Cross-platform workflow:**  
> Train in Colab → download `last.pt` + `args.yaml` → upload as Kaggle Dataset → resume in Kaggle → repeat.  
> The checkpoint format is identical in both notebooks.

## 1 · Environment Setup

In [1]:
"""
Section 1 · Environment Setup — Google Colab
Mounts GDrive, installs deps, detects GPU.
"""

from __future__ import annotations

ENV = "COLAB"
SESSION_TIME_LIMIT_H = 11.5   # Safe margin under Colab's 12h limit
print("=" * 60)
print(f"  [ENV DETECTED] {ENV}")
print(f"  [SESSION LIMIT] {SESSION_TIME_LIMIT_H} hours")
print("=" * 60)

!pip install -q ultralytics

import gc, json, os, shutil, time, warnings
from datetime import datetime, timezone
from pathlib import Path

# ── Mount Google Drive ──────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")
print("✅ Google Drive mounted.")

import cv2, matplotlib.pyplot as plt, numpy as np, pandas as pd
import seaborn as sns, torch, yaml
from IPython.display import Image as IPImage, display
from ultralytics import YOLO

warnings.filterwarnings("ignore", category=FutureWarning)

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM    : {total_vram:.1f} GB")

  [ENV DETECTED] COLAB
  [SESSION LIMIT] 11.5 hours
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 29.0 MB/s eta 0:00:00
Mounted at /content/drive
✅ Google Drive mounted.
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
PyTorch : 2.10.0+cu128
CUDA    : True
GPU     : Tesla T4
VRAM    : 15.6 GB


## 2 · Project Paths & Dataset

In [2]:
"""
Section 2a · Project Paths (Colab + GDrive)

Layout:
  GDrive: pothole_detection_system/backend/
    models/v8l/         ← best.pt, last.pt, args.yaml  (cross-platform checkpoint)
    models/checkpoints/ ← periodic sync copy of last.pt
    models/logs/        ← results.csv, training_log.json
    dataset_768/        ← training data (or dataset_768.tar)

  /content (VM local):
    dataset_768_cache/  ← fast local copy of dataset
"""
from pathlib import Path

BASE_DIR             = Path("/content")
GDRIVE_PROJECT_ROOT  = Path("/content/drive/MyDrive/pothole_detection_system")
BACKEND_DIR          = GDRIVE_PROJECT_ROOT / "backend"
PROJECT_ROOT         = GDRIVE_PROJECT_ROOT

DATASET_DIR_GDRIVE   = BACKEND_DIR / "dataset_768"
LOCAL_CACHE_DIR      = BASE_DIR / "dataset_768_cache"
DATASET_DIR          = LOCAL_CACHE_DIR
DATA_YAML            = DATASET_DIR / "data.yaml"

MODELS_DIR      = BACKEND_DIR / "models"
CHECKPOINTS_DIR = MODELS_DIR / "checkpoints"
FINAL_DIR       = MODELS_DIR / "final"
LOGS_DIR        = MODELS_DIR / "logs"
V8L_DIR         = MODELS_DIR / "v8l"          # ← Cross-platform checkpoint dir

INFERENCE_DIR = BACKEND_DIR / "inference_outputs"
RUNS_DIR      = BACKEND_DIR / "runs"
UTILS_DIR     = BACKEND_DIR / "utils"

for d in [MODELS_DIR, CHECKPOINTS_DIR, FINAL_DIR, LOGS_DIR,
          V8L_DIR, INFERENCE_DIR, RUNS_DIR, UTILS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("📁 Directory structure:")
for d in [MODELS_DIR, CHECKPOINTS_DIR, FINAL_DIR, LOGS_DIR,
          V8L_DIR, INFERENCE_DIR, RUNS_DIR, UTILS_DIR]:
    print("  ✅", d)

print(f"\n📦 Dataset source (GDrive): {DATASET_DIR_GDRIVE}")
print(f"📦 Dataset cache  (local) : {LOCAL_CACHE_DIR}")
assert DATASET_DIR_GDRIVE.exists(), f"❌ Dataset not found on GDrive at {DATASET_DIR_GDRIVE}"

📁 Directory structure:
  ✅ /content/drive/MyDrive/pothole_detection_system/backend/models
  ✅ /content/drive/MyDrive/pothole_detection_system/backend/models/checkpoints
  ✅ /content/drive/MyDrive/pothole_detection_system/backend/models/final
  ✅ /content/drive/MyDrive/pothole_detection_system/backend/models/logs
  ✅ /content/drive/MyDrive/pothole_detection_system/backend/models/v8l
  ✅ /content/drive/MyDrive/pothole_detection_system/backend/inference_outputs
  ✅ /content/drive/MyDrive/pothole_detection_system/backend/runs
  ✅ /content/drive/MyDrive/pothole_detection_system/backend/utils

📦 Dataset source (GDrive): /content/drive/MyDrive/pothole_detection_system/backend/dataset_768
📦 Dataset cache  (local) : /content/dataset_768_cache


In [3]:
"""
Section 2b · Smart Dataset Caching (Colab)

Copies dataset from GDrive to local /content for fast I/O during training.
1. Use dataset_768.tar if available (fastest)
2. Otherwise copy dataset folder from GDrive
3. Skip if cache already exists
"""
import time as _time

TAR_FILE_GDRIVE = DATASET_DIR_GDRIVE.parent / "dataset_768.tar"
LOCAL_TAR_FILE  = BASE_DIR / "dataset_768.tar"

if LOCAL_CACHE_DIR.exists() and (LOCAL_CACHE_DIR / "images").exists():
    print(f"✅ Dataset cache already exists at {LOCAL_CACHE_DIR}")
else:
    if LOCAL_CACHE_DIR.exists():
        shutil.rmtree(LOCAL_CACHE_DIR)

    if TAR_FILE_GDRIVE.exists():
        print("📦 TAR dataset detected → using fast extraction method")
        t0 = _time.time()
        !cp "{TAR_FILE_GDRIVE}" "{LOCAL_TAR_FILE}"
        print(f"✅ TAR copied in {_time.time()-t0:.1f}s")
        t1 = _time.time()
        !tar -xf "{LOCAL_TAR_FILE}" -C {BASE_DIR}/
        if (BASE_DIR / "dataset_768").exists():
            shutil.move(str(BASE_DIR / "dataset_768"), str(LOCAL_CACHE_DIR))
        print(f"✅ Dataset extracted in {_time.time()-t1:.1f}s")
    else:
        print("⚠️  TAR not found — copying folder from GDrive (slower)")
        t0 = _time.time()
        !cp -r "{DATASET_DIR_GDRIVE}" "{LOCAL_CACHE_DIR}"
        print(f"✅ Dataset copied in {_time.time()-t0:.1f}s")

DATA_YAML = LOCAL_CACHE_DIR / "data.yaml"
if not DATA_YAML.exists():
    raise FileNotFoundError(f"❌ data.yaml not found in {LOCAL_CACHE_DIR}")

with open(DATA_YAML, "r", encoding="utf-8") as f:
    data_cfg = yaml.safe_load(f)
data_cfg["path"] = str(LOCAL_CACHE_DIR.resolve())
with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.dump(data_cfg, f, default_flow_style=False, sort_keys=False)

print("\n📄 data.yaml updated:")
for k, v in data_cfg.items():
    print(f"  {k}: {v}")

_n_train = len(list((LOCAL_CACHE_DIR / "images" / "train").glob("*.*")))
_n_val   = len(list((LOCAL_CACHE_DIR / "images" / "val").glob("*.*")))
_n_test  = len(list((LOCAL_CACHE_DIR / "images" / "test").glob("*.*")))
print(f"\n📊 Cached images: train={_n_train}  val={_n_val}  test={_n_test}")
print("=" * 60)
print("  [DATASET READY] ✅")
print("=" * 60)

📦 TAR dataset detected → using fast extraction method
✅ TAR copied in 43.4s
✅ Dataset extracted in 23.4s

📄 data.yaml updated:
  path: /content/dataset_768_cache
  train: images/train
  val: images/val
  test: images/test
  nc: 1
  names: ['pothole']

📊 Cached images: train=13284  val=3796  test=1896
  [DATASET READY] ✅


In [4]:
"""
Section 2c · Resume Checkpoint Import (Colab ↔ Kaggle)

Looks for checkpoint files in V8L_DIR (models/v8l/).
These can be placed there:
  - Automatically by this notebook's sync_checkpoints()
  - Manually after downloading from a Kaggle session

Required files for full resume (resume=True):
  • last.pt   — full training state (epoch, optimizer, scheduler)
  • args.yaml — original training arguments

AUTO-PATCH: args.yaml coming from Kaggle has the Kaggle data path baked in
  (/kaggle/working/data.yaml). This cell automatically rewrites that path
  to the Colab local cache path so resume=True works without any manual edits.

If only best.pt is present:
  • Training will restart from epoch 0 using best weights
  • (you lose epoch count but keep learned features)
"""

COLAB_DATA_YAML_PATH = str(DATA_YAML.resolve())  # /content/dataset_768_cache/data.yaml

weights_dir = RUNS_DIR / "pothole_v8l" / "weights"

last_in_v8l = V8L_DIR / "last.pt"
args_in_v8l = V8L_DIR / "args.yaml"

if last_in_v8l.is_file():
    print(f"✅ Found last.pt in V8L_DIR: {last_in_v8l}")
    weights_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(last_in_v8l, weights_dir / "last.pt")
    print(f"   ↪ Copied to {weights_dir / 'last.pt'}")

    if args_in_v8l.is_file():
        # ── Auto-patch the data path in args.yaml ────────────────────────────
        # args.yaml from Kaggle has data: /kaggle/working/data.yaml
        # args.yaml from a previous Colab session is already correct
        # Either way, we patch it to the current Colab data.yaml path
        with open(args_in_v8l, "r", encoding="utf-8") as f:
            args_content = f.read()

        original_data_line = None
        patched_lines = []
        for line in args_content.splitlines():
            if line.strip().startswith("data:"):
                original_data_line = line.strip()
                patched_lines.append(f"data: {COLAB_DATA_YAML_PATH}")
            else:
                patched_lines.append(line)
        patched_content = "\n".join(patched_lines) + "\n"

        # Write patched args.yaml to the run dir
        run_args_path = RUNS_DIR / "pothole_v8l" / "args.yaml"
        run_args_path.parent.mkdir(parents=True, exist_ok=True)
        with open(run_args_path, "w", encoding="utf-8") as f:
            f.write(patched_content)

        if original_data_line and f"data: {COLAB_DATA_YAML_PATH}" not in original_data_line:
            print(f"   ↪ args.yaml patched:")
            print(f"       was : {original_data_line}")
            print(f"       now : data: {COLAB_DATA_YAML_PATH}")
        else:
            print(f"   ↪ args.yaml copied (data path already correct)")
    else:
        print("   ⚠️  args.yaml not found — resume=True will fall back to weight-only resume")
        print("        (this is fine — args.yaml is auto-generated after first 5-epoch sync)")

    if (V8L_DIR / "best.pt").is_file():
        shutil.copy2(V8L_DIR / "best.pt", weights_dir / "best.pt")
        print(f"   ↪ Copied best.pt to weights dir")
else:
    print("ℹ️  No last.pt found in V8L_DIR — training will start fresh or from best.pt.")

print("\n" + "=" * 60)
print("  [CHECKPOINT IMPORT] Complete")
print("=" * 60)

✅ Found last.pt in V8L_DIR: /content/drive/MyDrive/pothole_detection_system/backend/models/v8l/last.pt
   ↪ Copied to /content/drive/MyDrive/pothole_detection_system/backend/runs/pothole_v8l/weights/last.pt
   ⚠️  args.yaml not found — resume=True will fall back to weight-only resume
        (this is fine — args.yaml is auto-generated after first 5-epoch sync)
   ↪ Copied best.pt to weights dir

  [CHECKPOINT IMPORT] Complete


## 3 · GPU Detection & Dynamic Batch Sizing

In [5]:
"""
Dynamic GPU batch sizing via real memory profiling.

Targets 85% GPU VRAM (conservative for v8l which is larger than v8m).
Profiles YOLOv8l at 768px.
"""

GPU_TARGET_UTILIZATION = 0.85
MIN_BATCH   = 2
MAX_BATCH   = 64
IMGSZ_FALLBACKS = [768, 640, 512]


def _profile_model_memory(model_name: str, imgsz: int) -> tuple:
    """Profile GPU memory by running real forward passes.
    Returns: (base_memory_gb, per_image_memory_gb)"""
    torch.cuda.empty_cache()
    gc.collect()
    torch.cuda.reset_peak_memory_stats(0)

    model = YOLO(model_name)
    model.model.to("cuda").train()

    torch.cuda.reset_peak_memory_stats(0)
    dummy1 = torch.randn(1, 3, imgsz, imgsz, device="cuda", dtype=torch.float32)
    with torch.no_grad():
        _ = model.model(dummy1)
    mem_peak_b1 = torch.cuda.max_memory_allocated(0) / 1e9
    del dummy1, _
    torch.cuda.empty_cache()

    torch.cuda.reset_peak_memory_stats(0)
    dummy2 = torch.randn(2, 3, imgsz, imgsz, device="cuda", dtype=torch.float32)
    with torch.no_grad():
        _ = model.model(dummy2)
    mem_peak_b2 = torch.cuda.max_memory_allocated(0) / 1e9
    del dummy2, _

    per_image_mem = mem_peak_b2 - mem_peak_b1
    base_mem      = mem_peak_b1 - per_image_mem

    TRAINING_OVERHEAD = 2.2   # AMP disabled → use same overhead for static FP32
    per_image_mem *= TRAINING_OVERHEAD
    base_mem      *= TRAINING_OVERHEAD

    del model
    torch.cuda.empty_cache()
    gc.collect()

    if per_image_mem <= 0:
        per_image_mem = mem_peak_b1 * 0.4
        base_mem      = mem_peak_b1 * 0.6

    return max(base_mem, 0.5), max(per_image_mem, 0.05)


def estimate_batch_size(model_name: str, imgsz: int) -> tuple:
    """Dynamically estimate the optimal batch size.
    Returns: (batch_size, actual_imgsz)"""
    if not torch.cuda.is_available():
        print("⚠️  No GPU — defaulting to batch=2, CPU mode.")
        return MIN_BATCH, imgsz

    props = torch.cuda.get_device_properties(0)
    total_vram = props.total_memory / 1e9
    target_vram = total_vram * GPU_TARGET_UTILIZATION

    print(f"🖥️  GPU: {props.name}")
    print(f"   Total VRAM  : {total_vram:.2f} GB")
    print(f"   Target usage: {target_vram:.2f} GB ({GPU_TARGET_UTILIZATION*100:.0f}%)")

    sizes_to_try = [imgsz] + [s for s in IMGSZ_FALLBACKS if s < imgsz]

    for try_sz in sizes_to_try:
        try:
            print(f"\n📊 Profiling {model_name} @ {try_sz}px...")
            base_mem, per_img_mem = _profile_model_memory(model_name, try_sz)
            print(f"   Base memory   : {base_mem:.2f} GB")
            print(f"   Per-image mem : {per_img_mem:.3f} GB")

            available = target_vram - base_mem
            if available <= 0:
                print(f"   ⚠️  Model base memory exceeds target. Trying smaller imgsz.")
                continue

            optimal_batch = int(available / per_img_mem)
            batch = max(MIN_BATCH, min(MAX_BATCH, optimal_batch))

            predicted_usage = base_mem + batch * per_img_mem
            usage_pct = predicted_usage / total_vram * 100

            if batch >= MIN_BATCH:
                print(f"\n   ✅ Optimal batch size: {batch}")
                print(f"   Predicted VRAM usage: {predicted_usage:.2f} GB ({usage_pct:.1f}%)")
                if try_sz != imgsz:
                    print(f"   ⚠️  Reduced imgsz {imgsz} → {try_sz} to fit GPU.")
                return batch, try_sz
            else:
                print(f"   ⚠️  batch={optimal_batch} too small at {try_sz}px.")

        except Exception as e:
            print(f"   ⚠️  Profiling failed at {try_sz}px: {e}")
            torch.cuda.empty_cache()
            gc.collect()
            continue

    print("\n⚠️  All profiling attempts failed. Using conservative fallback.")
    return MIN_BATCH, 640


bs_test, sz_test = estimate_batch_size("yolov8l.pt", 768)
print("\n" + "=" * 50)
print(f"  Final: batch={bs_test}, imgsz={sz_test}")
print("=" * 50)

🖥️  GPU: Tesla T4
   Total VRAM  : 15.64 GB
   Target usage: 13.29 GB (85%)

📊 Profiling yolov8l.pt @ 768px...
   Base memory   : 0.50 GB
   Per-image mem : 0.347 GB

   ✅ Optimal batch size: 36
   Predicted VRAM usage: 12.98 GB (83.0%)

  Final: batch=36, imgsz=768


## 4 · Training Engine (Crash-Safe + Cross-Platform Resume)

In [ ]:
"""
Core training engine — YOLOv8l edition.

KEY FIXES vs the old v8m notebook:
  ① validate_checkpoint_health REMOVED
     • Root cause of the epoch-17 loop: AMP GradScaler state legitimately
       contains Inf values (growth_tracker, _scale). Checking the full
       checkpoint triggers false-positive "corrupted" detection, renames
       last.pt, and forces a restart from epoch 17 (the last synced copy).
     • Fix: skip health-check entirely. Let YOLO's own resume= handle it.
       If resume=True raises an exception we catch it and fall back to
       weight-only load — NO files are renamed or deleted.

  ② AMP disabled (amp=False)
     • AMP (mixed precision) is the root cause of NaN at epoch ~54.
       With a large model (v8l) at 768px and AdamW, the FP16 scaler
       occasionally overflows → Inf gradients → NaN loss.
     • FP32 training is ~10-15% slower but completely stable.

  ③ lr0 reduced to 3e-4 (was 5e-4 for v8m)
     • v8l has more parameters; lower LR prevents early loss spikes.

  ④ nbs=64 (nominal batch size)
     • Tells YOLO to scale LR/momentum relative to batch=64 regardless
       of actual batch size, keeping hyperparameters well-conditioned.

  ⑤ Resume cascade (NO file deletions)
     1. last.pt in run weights dir → resume=True
     2. last.pt in V8L_DIR (GDrive synced copy) → copy → resume=True
     3. best.pt in V8L_DIR → weight-only load (fresh epoch count)
     4. Nothing → fresh training from yolov8l.pt pretrained weights
"""

PIPELINE_START_TIME     = time.time()
CHECKPOINT_SYNC_INTERVAL = 5   # Sync to GDrive every N epochs


def elapsed_hours() -> float:
    return (time.time() - PIPELINE_START_TIME) / 3600


def find_resume_checkpoint(run_name: str, model_dir: Path) -> tuple:
    """
    Safe resume cascade — NO health checks, NO file renames/deletions.

    If a checkpoint is found, we attempt to use it. If YOLO raises an
    exception during model.train(resume=True), train_stage catches it
    and falls back to the next option automatically.

    Returns: (checkpoint_path | None, use_resume_flag)
    """
    weights_dir = RUNS_DIR / run_name / "weights"

    # 1. last.pt already in run weights dir (same session or re-run)
    last_pt = weights_dir / "last.pt"
    if last_pt.is_file():
        print("=" * 60)
        print(f"  [RESUME] last.pt found in weights dir")
        print(f"  Source: {last_pt}")
        print("=" * 60)
        return last_pt, True

    # 2. last.pt in model_dir (GDrive synced copy from prev session)
    last_pt_gdrive = model_dir / "last.pt"
    if last_pt_gdrive.is_file():
        dst = weights_dir
        dst.mkdir(parents=True, exist_ok=True)
        try:
            shutil.copy2(last_pt_gdrive, dst / "last.pt")
            # Copy args.yaml so YOLO knows original training config
            args_src = model_dir / "args.yaml"
            if args_src.is_file():
                shutil.copy2(args_src, RUNS_DIR / run_name / "args.yaml")
            print("=" * 60)
            print(f"  [RESUME] last.pt found in GDrive model dir → copied to weights dir")
            print(f"  Source: {last_pt_gdrive}")
            print("=" * 60)
            return dst / "last.pt", True
        except Exception as e:
            print(f"  ⚠️  Could not copy last.pt from GDrive: {e}")

    # 3. best.pt in model_dir (weight-only fallback — epoch count resets)
    best_pt = model_dir / "best.pt"
    if best_pt.is_file():
        print("=" * 60)
        print(f"  [RESUME] No last.pt found — using best.pt (weight-only, epoch resets)")
        print(f"  Source: {best_pt}")
        print("  Tip: All learned features are preserved; only epoch counter resets.")
        print("=" * 60)
        return best_pt, False

    # 4. Fresh start
    print("=" * 60)
    print("  [RESUME] No checkpoint found — starting fresh from yolov8l.pt")
    print("=" * 60)
    return None, False


def sync_checkpoints(run_name: str, model_dir: Path) -> None:
    """Copy best.pt + last.pt + args.yaml + results.csv to GDrive."""
    run_dir = RUNS_DIR / run_name
    weights = run_dir / "weights"
    if not weights.is_dir():
        return

    for src_name, dst_dir in [("best.pt", model_dir), ("last.pt", CHECKPOINTS_DIR)]:
        src = weights / src_name
        if src.is_file():
            try:
                shutil.copy2(src, dst_dir / src_name)
            except Exception as e:
                print(f"  ⚠️ Sync failed for {src_name}: {e}")

    # Always keep last.pt in V8L_DIR too (for cross-platform handoff)
    last_src = weights / "last.pt"
    if last_src.is_file():
        try:
            shutil.copy2(last_src, model_dir / "last.pt")
        except Exception:
            pass

    for csv_f in [run_dir / "results.csv"]:
        if csv_f.is_file():
            try:
                shutil.copy2(csv_f, LOGS_DIR / f"{run_name}_results.csv")
                shutil.copy2(csv_f, model_dir / "results.csv")
            except Exception:
                pass

    args_yaml = run_dir / "args.yaml"
    if args_yaml.is_file():
        try:
            shutil.copy2(args_yaml, model_dir / "args.yaml")
        except Exception:
            pass

    print(f"  [CHECKPOINT SAVED] ✅ Synced to GDrive ({model_dir.name}/)")


def copy_stage_artifacts(run_name: str, model_dir: Path) -> None:
    """Final artifact copy at stage completion."""
    run_dir = RUNS_DIR / run_name
    weights = run_dir / "weights"
    for src_name in ["best.pt", "last.pt"]:
        src = weights / src_name
        if src.is_file():
            shutil.copy2(src, model_dir / src_name)
            shutil.copy2(src, CHECKPOINTS_DIR / src_name)
            print(f"  ✅ {src_name} → {model_dir / src_name}")
    for pattern in ["*.csv", "*.png", "*.json"]:
        for f in run_dir.glob(pattern):
            shutil.copy2(f, LOGS_DIR / f"{run_name}_{f.name}")
    args_yaml = run_dir / "args.yaml"
    if args_yaml.is_file():
        shutil.copy2(args_yaml, model_dir / "args.yaml")
    print(f"  ✅ Logs → {LOGS_DIR}")


def train_stage(
    model_name: str, imgsz: int, epochs: int, run_name: str,
    model_dir: Path, patience: int = 30, max_retries: int = 3,
) -> dict | None:
    """
    Train YOLOv8l with:
      • Crash-safe resume (no checkpoint deletion)
      • AMP disabled (prevents NaN at epoch ~54)
      • Gradient clipping + NaN gradient sanitization
      • NaN loss guard (stops before writing a bad last.pt)
      • CUDA OOM recovery (halve batch → reduce imgsz)
      • Time guard (force-save before session limit)
      • Periodic GDrive sync every 5 epochs
    """
    batch, actual_imgsz = estimate_batch_size(model_name, imgsz)
    stage_start = time.time()

    for attempt in range(1, max_retries + 1):
        print("\n" + "=" * 65)
        print(f"🚀 {run_name} | Attempt {attempt}/{max_retries} | "
              f"batch={batch} | imgsz={actual_imgsz}")
        print(f"   Pipeline elapsed: {elapsed_hours():.1f} h | "
              f"Session limit: {SESSION_TIME_LIMIT_H} h")
        print("=" * 65 + "\n")

        try:
            ckpt, use_resume = find_resume_checkpoint(run_name, model_dir)

            # ── Gradient sanitization BEFORE optimizer step ──────────────────
            # Fires BEFORE the optimizer updates weights.
            # nan_to_num_ replaces NaN/Inf gradients with 0/clip values.
            # clip_grad_norm_ prevents gradient explosion even if a few
            # gradients escape nan_to_num_.
            def _grad_clip_callback(trainer):
                if hasattr(trainer, "model"):
                    for p in trainer.model.parameters():
                        if p.grad is not None:
                            torch.nan_to_num_(p.grad, nan=0.0, posinf=1e4, neginf=-1e4)
                    torch.nn.utils.clip_grad_norm_(
                        trainer.model.parameters(), max_norm=10.0
                    )

            # ── Epoch-end: NaN loss guard + sync + time guard ────────────────
            def _epoch_callback(trainer, _rn=run_name, _md=model_dir):
                # NaN loss guard — stop BEFORE YOLO writes a bad last.pt
                loss = getattr(trainer, "loss", None)
                if loss is not None:
                    try:
                        if not torch.isfinite(torch.as_tensor(loss)).all():
                            print(f"\n🚨 [NaN GUARD] NaN/Inf loss at epoch "
                                  f"{trainer.epoch} — stopping before checkpoint write!")
                            print("   This should not happen with amp=False. "
                                  "Check your dataset for corrupted labels.")
                            trainer.epoch = trainer.epochs  # force clean exit
                            return  # skip sync
                    except Exception:
                        pass

                # Periodic sync to GDrive
                if trainer.epoch % CHECKPOINT_SYNC_INTERVAL == 0 and trainer.epoch > 0:
                    sync_checkpoints(_rn, _md)

                # Time guard
                h = elapsed_hours()
                remaining = SESSION_TIME_LIMIT_H - h
                if remaining < 0.5:
                    print("\n" + "!" * 60)
                    print(f"  [TIME GUARD] ⚠️ Approaching session limit!")
                    print(f"  Elapsed: {h:.2f}h | Limit: {SESSION_TIME_LIMIT_H}h")
                    print(f"  Force-saving and stopping...")
                    print("!" * 60)
                    sync_checkpoints(_rn, _md)
                    trainer.epoch = trainer.epochs
                elif remaining < 1.0:
                    print(f"\n  ⏰ {remaining:.1f}h remaining before session limit")

            # ── Stable training hyperparameters for YOLOv8l ──────────────────
            # amp=False   → FP32 training; eliminates NaN/Inf from FP16 overflow
            # lr0=3e-4    → conservative for large model (v8l > v8m in params)
            # lrf=0.003   → proportional to lr0
            # warmup_epochs=10 → longer warmup stabilises large imgsz
            # nbs=64      → nominal batch size; LR/momentum scale correctly
            # clip_grad=10.0 → YOLO built-in gradient clip (belt-and-suspenders)
            # cache="ram" → fast data loading on Colab VM
            TRAIN_KWARGS = dict(
                data=str(DATA_YAML.resolve()),
                imgsz=actual_imgsz,
                epochs=epochs,
                patience=patience,
                batch=batch,
                optimizer="AdamW",
                lr0=3e-4,            # ← reduced vs v8m (5e-4) for stability
                lrf=0.003,           # ← proportional final LR
                cos_lr=True,
                weight_decay=5e-4,
                warmup_epochs=10,
                warmup_bias_lr=0.01,
                momentum=0.937,
                amp=False,           # ← CRITICAL: disables AMP → no NaN at epoch ~54 (for False)
                # clip_grad=10.0,
                nbs=64,              # ← nominal batch for correct LR scaling
                cache="ram",
                workers=4,
                device=0 if torch.cuda.is_available() else "cpu",
                hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
                degrees=5.0, translate=0.1, scale=0.5,
                shear=2.0, flipud=0.2, fliplr=0.5,
                mosaic=1.0, mixup=0.1, copy_paste=0.1,
                project=str(RUNS_DIR),
                name=run_name,
                exist_ok=True,
                save_period=5,
                plots=True,
                verbose=True,
            )

            if ckpt and use_resume:
                # Full resume — preserves epoch, optimizer state, scheduler
                print("🔄 Attempting full resume with resume=True...")
                try:
                    model = YOLO(str(ckpt))
                    model.add_callback("on_before_optimizer_step", _grad_clip_callback)
                    model.add_callback("on_train_epoch_end", _epoch_callback)
                    results = model.train(resume=True)
                except Exception as resume_err:
                    # If resume=True fails (e.g. args mismatch), fall back to weight-only
                    print(f"  ⚠️ resume=True failed: {resume_err}")
                    print("  ↪ Falling back to weight-only load (epoch resets)...")
                    torch.cuda.empty_cache()
                    gc.collect()
                    model = YOLO(str(ckpt))
                    model.add_callback("on_before_optimizer_step", _grad_clip_callback)
                    model.add_callback("on_train_epoch_end", _epoch_callback)
                    results = model.train(**TRAIN_KWARGS)

            elif ckpt and not use_resume:
                # Weight-only resume (best.pt) — fresh training with loaded weights
                print("🔄 Loading best.pt weights (weight-only, epoch resets to 0)...")
                model = YOLO(str(ckpt))
                model.add_callback("on_before_optimizer_step", _grad_clip_callback)
                model.add_callback("on_train_epoch_end", _epoch_callback)
                results = model.train(**TRAIN_KWARGS)

            else:
                # Fresh start from pretrained weights
                print("✨ Starting fresh training from yolov8l.pt pretrained weights...")
                model = YOLO(model_name)
                model.add_callback("on_before_optimizer_step", _grad_clip_callback)
                model.add_callback("on_train_epoch_end", _epoch_callback)
                results = model.train(**TRAIN_KWARGS)

            # ── Training completed successfully ──────────────────────────────
            duration = (time.time() - stage_start) / 3600
            print(f"\n✅ {run_name} completed in {duration:.2f} hours.")
            copy_stage_artifacts(run_name, model_dir)

            log_entry = {
                "stage": run_name, "model": model_name, "imgsz": actual_imgsz,
                "batch": batch, "duration_hours": round(duration, 2),
                "timestamp": datetime.now(timezone.utc).isoformat(),
            }
            log_file = LOGS_DIR / "training_log.json"
            logs = json.loads(log_file.read_text()) if log_file.is_file() else []
            logs.append(log_entry)
            log_file.write_text(json.dumps(logs, indent=2))

            best_pt = model_dir / "best.pt"
            if best_pt.is_file():
                val_model = YOLO(str(best_pt))
                metrics = val_model.val(
                    data=str(DATA_YAML.resolve()), imgsz=actual_imgsz,
                    batch=8, device=0 if torch.cuda.is_available() else "cpu",
                    verbose=False,
                )
                return {
                    "stage": run_name, "model": model_name, "imgsz": actual_imgsz,
                    "mAP50": metrics.box.map50, "mAP50_95": metrics.box.map,
                    "precision": metrics.box.mp, "recall": metrics.box.mr,
                    "duration_h": round(duration, 2),
                }
            return None

        except RuntimeError as e:
            err_str = str(e).lower()
            if "out of memory" in err_str or ("cuda" in err_str and "nan" not in err_str):
                print(f"\n⚠️  CUDA OOM at batch={batch}, imgsz={actual_imgsz}")
                torch.cuda.empty_cache()
                gc.collect()
                if batch > MIN_BATCH:
                    batch = max(MIN_BATCH, batch // 2)
                    print(f"   → Reducing batch to {batch}")
                elif actual_imgsz > 640:
                    actual_imgsz -= 64
                    batch = MIN_BATCH
                    print(f"   → Reducing imgsz to {actual_imgsz}, batch={batch}")
                else:
                    print("   ❌ Cannot reduce further. Skipping.")
                    return None
            else:
                print(f"\n❌ Unhandled RuntimeError: {e}")
                raise

    print(f"\n❌ {run_name}: All {max_retries} attempts exhausted.")
    return None


print("✅ Training engine loaded (YOLOv8l — NaN-free edition).")
print(f"   AMP             : DISABLED (amp=False) — eliminates NaN/Inf")
print(f"   Learning rate   : lr0=3e-4 (conservative for large model)")
print(f"   Gradient clip   : max_norm=10.0 + nan_to_num_ on gradients")
print(f"   NaN loss guard  : stops training before bad checkpoint write")
print(f"   Resume strategy : NO health-check, NO file deletion — safe cascade")
print(f"   Checkpoint sync : every {CHECKPOINT_SYNC_INTERVAL} epochs")
print(f"   Time guard      : {SESSION_TIME_LIMIT_H} hours")

✅ Training engine loaded (YOLOv8l — NaN-free edition).
   AMP             : DISABLED (amp=False) — eliminates NaN/Inf
   Learning rate   : lr0=3e-4 (conservative for large model)
   Gradient clip   : max_norm=10.0 + nan_to_num_ on gradients
   NaN loss guard  : stops training before bad checkpoint write
   Resume strategy : NO health-check, NO file deletion — safe cascade
   Checkpoint sync : every 5 epochs
   Time guard      : 11.5 hours


## 5 · Train YOLOv8l (768px, 400 epochs)

In [7]:
# Quick sanity check before starting
import torch
print("Torch:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

from ultralytics import YOLO
YOLO("yolov8n.pt")   # tiny download to verify ultralytics is working
print("Ultralytics ✅")

Torch: 2.10.0+cu128
CUDA Available: True
GPU: Tesla T4
Ultralytics ✅


In [ ]:
"""
YOLOv8l Training — 768px · 400 epochs

This cell is safe to re-run after a session crash.
It will automatically resume from the last checkpoint on GDrive.

Cross-platform workflow:
  Colab  → train → checkpoints synced to GDrive (models/v8l/)
  Kaggle → download last.pt+args.yaml from GDrive → upload as Kaggle Dataset
         → Section 2c of Kaggle notebook imports them → resume=True
  Colab  → download last.pt+args.yaml from Kaggle output → place in GDrive models/v8l/
         → re-run this cell → resume=True
"""

print("\n" + "━" * 65)
print("  TRAINING — YOLOv8l @ 768px")
print("━" * 65 + "\n")

metrics_v8l = train_stage(
    model_name = "yolov8l.pt",
    imgsz      = 768,
    epochs     = 400,
    run_name   = "pothole_v8l",
    model_dir  = V8L_DIR,
    patience   = 30,
)

if metrics_v8l:
    print("\n📊 Training Results:")
    for k, v in metrics_v8l.items():
        print(f"  {k:12s}: {v}")
else:
    print("\n⚠️  Training stopped early (time guard or session limit).")
    print("   Checkpoint is saved. Re-run this cell or continue in Kaggle.")

torch.cuda.empty_cache()
gc.collect()
print(f"\n⏱️  Total pipeline time: {elapsed_hours():.1f} h")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  TRAINING — YOLOv8l @ 768px
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🖥️  GPU: Tesla T4
   Total VRAM  : 15.64 GB
   Target usage: 13.29 GB (85%)

📊 Profiling yolov8l.pt @ 768px...
   Base memory   : 0.50 GB
   Per-image mem : 0.347 GB

   ✅ Optimal batch size: 36
   Predicted VRAM usage: 12.98 GB (83.0%)

🚀 pothole_v8l | Attempt 1/3 | batch=36 | imgsz=768
   Pipeline elapsed: 0.0 h | Session limit: 11.5 h

  [RESUME] last.pt found in weights dir
  Source: /content/drive/MyDrive/pothole_detection_system/backend/runs/pothole_v8l/weights/last.pt
🔄 Attempting full resume with resume=True...
Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=18, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.1

## 6 · Evaluation & Visualizations

In [ ]:
"""
Copy best model to final/ and display metrics.
Can be run after training or after a fresh session loading a saved model.
"""

print("\n" + "=" * 65)
print("  MODEL EVALUATION")
print("=" * 65 + "\n")

metrics_v8l_local = globals().get("metrics_v8l", None)

if metrics_v8l_local is None and (V8L_DIR / "best.pt").is_file():
    print("⚠️  Recovering metrics from saved best.pt...")
    try:
        val_model = YOLO(str(V8L_DIR / "best.pt"))
        metrics = val_model.val(
            data=str(DATA_YAML.resolve()), imgsz=768, batch=8,
            device=0 if torch.cuda.is_available() else "cpu", verbose=False,
        )
        metrics_v8l_local = {
            "stage": "pothole_v8l", "model": "yolov8l.pt", "imgsz": 768,
            "mAP50": metrics.box.map50, "mAP50_95": metrics.box.map,
            "precision": metrics.box.mp, "recall": metrics.box.mr,
            "duration_h": -1,
        }
    except Exception as e:
        print(f"  ⚠️ Recovery failed: {e}")

if metrics_v8l_local:
    print("📊 YOLOv8l Results:")
    for key in ["mAP50", "mAP50_95", "precision", "recall", "duration_h", "imgsz"]:
        val = metrics_v8l_local.get(key, "N/A")
        if isinstance(val, float):
            print(f"  {key:>15s}: {val:.4f}")
        else:
            print(f"  {key:>15s}: {val}")

# Copy best model to final/
src = V8L_DIR / "best.pt"
if src.is_file():
    shutil.copy2(src, FINAL_DIR / "best.pt")
    print(f"\n✅ Best model → {FINAL_DIR / 'best.pt'}")

BEST_MODEL_PATH = FINAL_DIR / "best.pt"
if not BEST_MODEL_PATH.is_file():
    BEST_MODEL_PATH = V8L_DIR / "best.pt"

print(f"\n🏆 BEST_MODEL_PATH = {BEST_MODEL_PATH} (exists: {BEST_MODEL_PATH.is_file()})")

In [ ]:
"""
Plot training curves from results.csv.
"""

run_name = "pothole_v8l"
csv_path = RUNS_DIR / run_name / "results.csv"

if not csv_path.is_file():
    # Try GDrive log copy
    csv_path = LOGS_DIR / f"{run_name}_results.csv"

if csv_path.is_file():
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    metrics_to_plot = [
        ("metrics/mAP50(B)",    "mAP@50"),
        ("metrics/mAP50-95(B)", "mAP@50-95"),
        ("train/box_loss",      "Train Box Loss"),
        ("val/box_loss",        "Val Box Loss"),
        ("train/cls_loss",      "Train Cls Loss"),
        ("metrics/precision(B)", "Precision"),
    ]

    available = [(col, label) for col, label in metrics_to_plot if col in df.columns]
    if available:
        fig, axes = plt.subplots(2, 3, figsize=(15, 8))
        axes = axes.flatten()
        for ax, (col, label) in zip(axes, available):
            ax.plot(df["epoch"] if "epoch" in df.columns else df.index, df[col])
            ax.set_title(label)
            ax.set_xlabel("Epoch")
            ax.grid(True, alpha=0.3)
        for ax in axes[len(available):]:
            ax.set_visible(False)
        plt.suptitle("YOLOv8l Training Progress", fontsize=14, fontweight="bold")
        plt.tight_layout()
        plt.savefig(str(LOGS_DIR / "v8l_training_curves.png"), dpi=150, bbox_inches="tight")
        plt.show()
        print("✅ Training curves plotted.")
    else:
        print(f"⚠️  Expected metric columns not found. Available: {list(df.columns)}")
else:
    print(f"ℹ️  results.csv not found at {csv_path}")
    print("   Run the training cell first or place results.csv in LOGS_DIR.")

## 7 · Cross-Platform Handoff Guide (Colab ↔ Kaggle)

### Moving from Colab → Kaggle

1. After training (or when Colab session ends), go to your Google Drive:  
   `pothole_detection_system/backend/models/v8l/`

2. Download these files:
   - `last.pt` — full training state (epoch number, optimizer, scheduler)
   - `args.yaml` — original training arguments (needed for `resume=True`)
   - `best.pt` — best weights so far (optional but good to have)
   - `results.csv` — training curve data

3. In Kaggle, create a new **Dataset** (or version an existing one) with these files.

4. In the Kaggle notebook, attach that dataset via **Add Data**.

5. The Kaggle notebook's Section 2c will auto-detect and import the checkpoint.

### Moving from Kaggle → Colab

1. After your Kaggle session finishes, go to the notebook **Output** tab.

2. Download `last.pt`, `args.yaml`, `best.pt`, `results.csv`.

3. Upload them to Google Drive at:  
   `pothole_detection_system/backend/models/v8l/`

4. In Colab, run Section 2c (`Resume Checkpoint Import`) and then Section 5 (Train).

> **Note:** `resume=True` requires both `last.pt` **and** `args.yaml`.  
> Without `args.yaml`, the notebook falls back to weight-only resume automatically.

## 8 · Inference

In [ ]:
"""
Run inference on a single image or folder.
"""

BEST_MODEL_PATH = FINAL_DIR / "best.pt"
if not BEST_MODEL_PATH.is_file():
    BEST_MODEL_PATH = V8L_DIR / "best.pt"

# ── Set your image path here ────────────────────────────────────────
IMAGE_PATH = None  # e.g. "/content/drive/MyDrive/test_image.jpg"
CONF_THRESH = 0.25

if IMAGE_PATH and Path(IMAGE_PATH).exists():
    model = YOLO(str(BEST_MODEL_PATH))
    results = model.predict(
        source=IMAGE_PATH, imgsz=768, conf=CONF_THRESH,
        device=0 if torch.cuda.is_available() else "cpu",
        save=True, project=str(INFERENCE_DIR), name="predict",
    )
    r = results[0]
    print(f"Detected {len(r.boxes)} pothole(s)")
    out_img = r.plot()
    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(out_img, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title(f"YOLOv8l Detections (conf>{CONF_THRESH})")
    plt.show()
else:
    print("ℹ️  Set IMAGE_PATH above to run inference.")
    print(f"   Best model: {BEST_MODEL_PATH} (exists: {BEST_MODEL_PATH.is_file()})")

## 9 · Export & Utilities

In [ ]:
"""
Export best model to ONNX for deployment.
"""

def export_onnx(model_path: Path = None) -> None:
    if model_path is None:
        model_path = FINAL_DIR / "best.pt"
        if not model_path.is_file():
            model_path = V8L_DIR / "best.pt"
    if not model_path.is_file():
        print(f"❌ No model found at {model_path}")
        return
    m = YOLO(str(model_path))
    # Note: half=True requires CUDA; set half=False for CPU export
    m.export(format="onnx", imgsz=768, half=False, simplify=True)
    print(f"✅ ONNX exported next to {model_path}")

# Uncomment to export:
# export_onnx()

In [ ]:
"""
Pipeline summary.
"""

try:
    total_h = elapsed_hours()
except NameError:
    total_h = -1

print("\n" + "=" * 65)
print("  📋 PIPELINE SUMMARY")
print("=" * 65)
print(f"  Total time  : {total_h:.2f} hours" if total_h >= 0 else "  Total time  : unknown")
print(f"  V8L dir     : {V8L_DIR}")
print(f"  Final dir   : {FINAL_DIR}")
print(f"  Logs dir    : {LOGS_DIR}")

for split in ["train", "val", "test"]:
    d = LOCAL_CACHE_DIR / "images" / split
    n = len(list(d.glob("*"))) if d.is_dir() else 0
    print(f"  Dataset {split:>5}: {n:,} images")

log_file = LOGS_DIR / "training_log.json"
if log_file.is_file():
    print("\n  Training log:")
    for entry in json.loads(log_file.read_text()):
        print(
            f"    {entry['stage']:25s} | "
            f"{entry['duration_hours']:.2f} h | "
            f"imgsz={entry['imgsz']} batch={entry['batch']}"
        )

print("=" * 65)